In [1]:
import json
from ACL_cycle import run_llm_batfish_gns3_cycle

from Helpers.Parse import parse_config_to_json
from Helpers.Read_Files import read_all_json_files_in_folder_Eval



In [2]:
#################### Device information to access GNS3 server: consol IP and port ####################
######################################################################################################
Consol = [ # this info extracted from GNS3 consol (they are based on the VM server)
    {
        "D_Name" : "PC1",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5012"
    },
    {
        "D_Name" : "PC2",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5014"
    },
    {
        "D_Name" : "PC3",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5016"
    },
    {
        "D_Name" : "PC4",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5018"
    },
    {
        "D_Name" : "PC5",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5020"
    },
    {
        "D_Name" : "PC6",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5022"
    },
    {
        "D_Name" : "R1",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5006"
    },
    {
        "D_Name" : "R2",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5007"
    },
    {
        "D_Name" : "R3",
        "D_IP"   : "10.133.14.34",
        "D_Port" : "5008"
    }
]

for d in Consol:
    if d["D_Name"].startswith("PC"):
        d["Type"] = "VPCS"
    else:
        d["Type"] = "Router"

Sub_Net = [ # this info extracted from GNS3 topology
    {
        "D_Name" : "PC1",
        "D_IP"   : "172.16.10.2",
        "D_Net" : "172.16.10.0/24"
    },
    {
        "D_Name" : "PC2",
        "D_IP"   : "172.16.10.3",
        "D_Net" : "172.16.10.0/24"
    },
    {
        "D_Name" : "PC3",
        "D_IP"   : "172.16.30.2",
        "D_Net" : "172.16.30.0/24"
    },
    {
        "D_Name" : "PC4",
        "D_IP"   : "172.16.30.3",
        "D_Net" : "172.16.30.0/24"
    },
    {
        "D_Name" : "PC5",
        "D_IP"   : "172.16.50.2",
        "D_Net" : "172.16.50.0/24"
    },
    {
        "D_Name" : "PC6",
        "D_IP"   : "172.16.50.3",
        "D_Net" : "172.16.50.0/24"
    }
]

In [3]:
############# Set input variables + default variables to the system: intent, etc ##############
################################################################################################

parse_config_to_json('configs/R1')#.cfg')
parse_config_to_json('configs/R2')#.cfg')
parse_config_to_json('configs/R3')#.cfg')

new_intent = input("Hi!\n Type your intent here : ").strip().lower()    
file_path           = "./configs/" #"./configs"  # Replace with your folder path
file_contents       = read_all_json_files_in_folder_Eval(file_path)
network_topology            = "./configs/TopologyFile.json"
# new_intent          = "Allow PC1 to SSH to WEB1 in the DMZ"
# new_intent  = "Block HTTP traffic from subnet 172.16.10.0/24 to subnet 172.16.50.0/24" #correct
# new_intent  = "Block HTTP traffic from LAN10 to subnet LAN50" #correct

# # Eval_GT_path             = "./Groundtruth/Multirouter_generated/multirouter_intents_mixed.json"
# Eval_GT_path             = "./Groundtruth/Multirouter_generated/multirouter_intents_mixed_sample.json"
# Conflict_GT_PATH         = "./Groundtruth/acl_conflict_detection_GT.json"

######  Used GT for Evaluation LLM Performance  ######

# file_path           = "./configs"  # Replace with your folder path
# # file_path           = "./configs"  # Replace with your folder path
# file_contents       = read_all_json_files_in_folder(file_path)
# # Topology            = "./configs/TopologyFile.json"

# Topology            = "./configs/TopologyFile.json"
# new_intent          = "Deny the Internet from accessing the LAN network"
# Eval_GT             = "./Groundtruth/easy-cisco-Intents2.json"

######## Load dataset for eval #######
with open(network_topology, "r", encoding="utf-8") as f:
    network_topology_json = json.load(f) # for eval
############# Initial values #############
topology_file       = None
hostname            = None
Whole_configuration = None
extraction_result   = None
L_Name              = None
Rules               = None
direction           = None
Intf_Name           = None
user_action         = None
List_Found          = None

context_variables = {
              "new_intent" : new_intent,
              "file_path"  : file_path,
           "topology_file" : file_contents,
        "network_topology" : network_topology    
    #   "network_topology" : network_topology_json,
    #   "Eval_GT_File"     : Eval_GT
}


Hi!
 Type your intent here :  block http traffic from subnet 172.16.10.0/24 to subnet 172.16.50.0/24


In [5]:
context_variables = {
    "new_intent": new_intent,
    "file_path": file_path,
    "topology_file": file_contents,
    "network_topology": network_topology_json,   
    "Sub_Net": Sub_Net,
    "devices": Consol,
    "snapshot_name": "configs",
    "enable_conflict_detection": True,
}

result = run_llm_batfish_gns3_cycle(
    context_variables=context_variables,
    batfish_host="10.133.14.31",   # or your Batfish server IP
    devices=Consol,
    max_gns3_repairs=3,
    max_batfish_iters=3,
    include_save=True,
    cleanup_candidate_on_success=False,
    verbose=True,
)

    
print(result["status"])


======== INITIAL LLM GENERATION ========
new_intent = 'block http traffic from subnet 172.16.10.0/24 to subnet 172.16.50.0/24'


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


=== DEVICE DEBUG ===
source_ip: None
src_subnet: None
dst_ip: None
dst_subnet: 172.16.10.0/24
wan_edge_router: None
router_networks: {'r1': ['172.16.10.0/24', '172.16.20.0/24'], 'r2': ['172.16.30.0/24', '172.16.20.0/24', '172.16.40.0/24'], 'r3': ['172.16.50.0/24', '172.16.40.0/24']}
[GPT:gpt-40] Processing: 'Topology (JSON):\n        {\n  "objects": {\n    "LAN10": "172.16.10.0/24",\n    "PC1": "172.16.10.2/32",\n    "PC2": "172.16' ...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[GPT RAW OUTPUT]: 'f0/1'
[GPT:gpt-40] Processing: 'Intent:\n        block http traffic from subnet 172.16.10.0/24 to subnet 172.16.50.0/24\n        \n        Chosen device:\n ' ...


...


=== CFG BEFORE BATFISH ===
!

!
version 12.4
service timestamps debug datetime msec
service timestamps log datetime msec
no service password-encryption
!
hostname R1_2
!
boot-start-marker
boot-end-marker
!
!
no aaa new-model
memory-size iomem 5
no ip icmp rate-limit unreachable
ip cef
!
!
!
!
!
multilink bundle-name authenticated
!
!
!
!
!
!
!
!
!
!
!
!
!
!
!
!
!
!
!
!
!
archive
 log config
  hidekeys
! 
!
!
!
ip tcp synwait-time 5
!
!
!
!

ip access-list extended ACL_F0_1_IN
deny tcp any 172.16.10.0 0.0.0.255 eq 80
!
!
ip access-list extended ACL_f0/1_IN
deny tcp any 172.16.10.0 0.0.0.255 eq 80
!
!
ip access-list extended ACL_F0/1_IN
deny tcp any 172.16.10.0 0.0.0.255 eq 80
!
!
interface FastEthernet0/0
 no ip address
 ip nat inside
 ip virtual-reassembly
 duplex auto
 speed auto
!
interface Serial0/0
 ip address 172.16.20.1 255.255.255.0
 clock rate 2000000
!
interface FastEthernet0/1
 ip address 172.16.10.1 255.255.255.0
 ip nat inside
 ip virtual-reassembly
 duplex auto
 speed aut

INFO:pybatfish.client.workhelper:status: WorkStatusCode.ASSIGNED
INFO:pybatfish.client.workhelper:.... no task information
INFO:pybatfish.client.workhelper:status: WorkStatusCode.ASSIGNED
INFO:pybatfish.client.workhelper:.... 2026-05-05 14:37:45.183000+02:00 Deserializing objects of type 'org.batfish.datamodel.Configuration' from files 5 / 5.
INFO:pybatfish.client.workhelper:status: WorkStatusCode.TERMINATEDNORMALLY
INFO:pybatfish.client.workhelper:.... 2026-05-05 14:37:45.183000+02:00 Deserializing objects of type 'org.batfish.datamodel.Configuration' from files 5 / 5.
INFO:pybatfish.client.session:Default snapshot is now set to configs
INFO:pybatfish.client.workhelper:status: WorkStatusCode.ASSIGNED
INFO:pybatfish.client.workhelper:.... no task information
INFO:pybatfish.client.workhelper:status: WorkStatusCode.TERMINATEDNORMALLY
INFO:pybatfish.client.workhelper:.... 2026-05-05 14:37:45.449000+02:00 Parse environment BGP tables.
INFO:pybatfish.client.workhelper:status: WorkStatusCode


=== CFG BEFORE BATFISH ===
!

!
version 12.4
service timestamps debug datetime msec
service timestamps log datetime msec
no service password-encryption
!
hostname R1_2
!
boot-start-marker
boot-end-marker
!
!
no aaa new-model
memory-size iomem 5
no ip icmp rate-limit unreachable
ip cef
!
!
!
!
!
multilink bundle-name authenticated
!
!
!
!
!
!
!
!
!
!
!
!
!
!
!
!
!
!
!
!
!
archive
 log config
  hidekeys
! 
!
!
!
ip tcp synwait-time 5
!
!
!
!

ip access-list extended ACL_F0_1_IN
deny tcp any 172.16.10.0 0.0.0.255 eq 80
!
!
ip access-list extended ACL_f0/1_IN
deny tcp any 172.16.10.0 0.0.0.255 eq 80
!
!
ip access-list extended ACL_F0/1_IN
deny tcp any 172.16.10.0 0.0.0.255 eq 80
!
!
interface FastEthernet0/0
 no ip address
 ip nat inside
 ip virtual-reassembly
 duplex auto
 speed auto
!
interface Serial0/0
 ip address 172.16.20.1 255.255.255.0
 clock rate 2000000
!
interface FastEthernet0/1
 ip address 172.16.10.1 255.255.255.0
 ip nat inside
 ip virtual-reassembly
 duplex auto
 speed aut

INFO:pybatfish.client.workhelper:status: WorkStatusCode.TERMINATEDNORMALLY
INFO:pybatfish.client.workhelper:.... 2026-05-05 14:37:45.564000+02:00 Begin job.
INFO:pybatfish.client.workhelper:status: WorkStatusCode.ASSIGNED
INFO:pybatfish.client.workhelper:.... no task information
INFO:pybatfish.client.workhelper:status: WorkStatusCode.TERMINATEDNORMALLY
INFO:pybatfish.client.workhelper:.... 2026-05-05 14:37:45.677000+02:00 Begin job.


run_result status: ran
top_status: ran
final, final_status
{'status': 'ok', 'stage': None, 'summary': 'OK', 'reasons': []} ok
reason: None
Saved candidate config: ./configs/R1_2.cfg -> ./configs/R1.cfg
Rewrote hostname: R1_2 -> R1
VERIFY OUT:

$ terminal length 0


$ show access-lists


$ show running-config | include access-group

PRECHECK OUT:

$ terminal length 0


$ show ip interface brief


$ show ip route

gns3_technical_failure


In [3]:
########### Import Libraries ###########
########################################

import os
import re
import sys
import json
import time
import random
import getpass
import telnetlib
import ipaddress
import traceback
import subprocess
import pandas as pd
from pprint import pprint
from tabulate import tabulate

from swarm import Swarm, Agent

import Helpers.env
from Helpers.Formats import wrap_text, wrap_dataframe, extract_table_info, parse_kv_response, csv_to_list,to_none_if_noneish, _norm_nl
from Helpers.Read_Files import read_all_files_in_folder, read_all_json_files_in_folder, read_all_vpc_files_in_folder, read_topology_file, read_all_json_files_in_folder_Eval
from Helpers.Parse import parse_config_to_json, resolve_names, ensure_topology_dict 
from Helpers.Finder import find_pcs_in_same_network, get_device_info, get_pc_name_by_ip, extract_router_facts_from_json, choose_device_only
from Helpers.RuleConflict import main_conflictDetection
from Helpers.configs import process_configuration, generate_config2_file, generate_config2_file_apply_only, add_updated_router_cmd, extract_rule_and_ACL_lines

from Agents.multiAgent import run_ACL_workflow
from Agents.Agents import * #client, get_direction, Answer_Query, File_Finder_caller, ACL_generator_caller, Entity_Extractor_caller

# noinspection PyUnresolvedReferences
# from pybatfish.client.session import Session
# from pybatfish.datamodel import Edge, Interface  
# from pybatfish.datamodel.answer import TableAnswer
# from pybatfish.datamodel.flow import HeaderConstraints, PathConstraints  
# from pybatfish.datamodel.route import BgpRoute  
# from pybatfish.util import get_html

from Batfish.Batfish import *
from Batfish.preprocess import *
from Batfish.preprocess import _intent_directionality, _infer_stage_from_results
from Batfish.Questions import *

# from Evaluation.Helper import *
# from Evaluation.Conf_DetectorEval import *
# from Evaluation.Evaluate import evaluate_entity_extractor, run_full_entity_eval, export_entity_eval_excel, evaluate_choose_device_only_deterministic, evaluate_query_agent_all, evaluate_acl_generator_all, append_device_eval_to_excel, evaluate_end_to_end
# import Agents.Agents

from GNS3.Validate import *
from GNS3.GNS3 import apply_config_GNS3, Router_Access

# %run startup.py

In [6]:
# ################################################## Program Workflow  #################################################################
context_variables = {
       "new_intent" : new_intent,
       "file_path"  : file_path,
    "topology_file" : file_contents,
}
Rules, hostname, topology_file, configuration_response, Whole_configuration, extraction_result, ACLname, direction, Intf_Name, File_name, List_Found = run_ACL_workflow(context_variables = context_variables)
##### Now go to stage 1 : Validation using BF #####

##### if return success --> go to stage 2: Validation using GNS3 #####


new_intent = 'allow pc1 to ssh to web1 in dmz'


TypeError: Unsupported network_topology type: <class 'NoneType'>

In [6]:
# # ################################################## Program Workflow  #################################################################
# # Read Intents --> Resolve_names --> (1) entity extraction Agent    -->  Select_device (func)
# # 
# # (2)Rule Conflict Agent  --> (3) Query Agent (many times) --> (4) ACL generator Agent --> (5)generate updated config 
# #
# # (6) Stage1: BF validation --> (7) Finetuning (if needed) --> (8) Stage2: GNS3 validation 
# #
# # --> (9) Finetuning (if needed) (10) Deployement
# # #######################################################################################################################################

# def run_ACL_workflow(context_variables = context_variables):
#     topology_file      = context_variables.get("topology_file")          # folder path containing .json
#     network_topology   = context_variables.get("network_topology")          # folder path containing .json
#     new_intent         = context_variables.get("new_intent")          # folder path containing .json
#     Eval_GT_File       = context_variables.get("Eval_GT_File")

#     print("User intent is : " + new_intent)
    
#     ############ resolve_names : mapping the obj and IPs ############
#     topo = ensure_topology_dict(network_topology)
#     # print("topo")
#     # print(topo)
#     resolved_names = resolve_names(new_intent, topo)
#     # print("resolved_names")
#     # print(resolved_names)

#     context_variables.update({
#           "resolved_names" : resolved_names,  
#                     })

#     # print(topology_file,network_topology,new_intent,Eval_GT_File)
    
#     ############## Step 1: EntityExtraction Agent - extract entities of the user intent ##############
#     # extracts entities from user intent
#     # uses resolved endpoints as a reference
#     # normalizes protocols, ports, and actions
#     # output --> structured 
#     ###############################################################################################################
       
#     # extraction_result = Entity_Extractor_caller(context_variables)
#     # # print(extraction_result)
#     # source_ip, destination_ip, protocol, port, action, app, src_Subnet, dst_Subnet = extract_entities(extraction_result)
#     # print(source_ip, destination_ip, protocol, port, action, app, src_Subnet, dst_Subnet)

#     extraction_result = Entity_Extractor_Evalcaller(context_variables)
#     # print("extraction_result")
#     # print(extraction_result)
#     source_ip, destination_ip, protocol, port, action, app, src_Subnet, dst_Subnet = extract_entities(extraction_result)
#     if src_Subnet == "0.0.0.0/0":
#         source_ip = None
#         src_Subnet = None
    
#     if dst_Subnet == "0.0.0.0/0":
#         destination_ip = None
#         dst_Subnet = None
    
#     # print(source_ip, destination_ip, protocol, port, action, app, src_Subnet, dst_Subnet)

    
#     ########################### Step 2: Find the related Device (python-based module) ##########
#     ######################################################################################
#     device_override = extract_device_override(new_intent)
#     if device_override:
#         hostname = device_override
#     else:
#         hostname = choose_device_only(source_ip, destination_ip, src_Subnet, dst_Subnet, network_topology,intent=new_intent)

#     print("hostname:", hostname)

#     File_name = hostname            # when test the main

#     cfg_path = f"{file_path}/{File_name}.cfg"
    
#     topology_content = read_topology_file(cfg_path) 
#     if isinstance(topology_content, list):
#         config_text = "".join(topology_content)
#         # print(config_text)
#     else:
#         config_text = topology_content or ""
        
#     ########################### Step 3: Rule Equivalence Validator Agent: Does an equivalent rule already exist in the router config? ##########################
#     #  semantic equivalence check
#     #  duplicate rule prevention
#     #  attribute comparison
#     #  binary decision
#     #########################################################################################################################################
    
#     conflict_q = (
#         "You are checking a Cisco IOS configuration for an existing ACL rule.\n"
#         f"Intent: {new_intent}\n\n"
#         "Return ONLY one token:\n"
#         "True  -> if an ACL line exists in the configuration that is semantically equivalent to the intent. "
#         "(same action + protocol + src + dst + service/port).\n"
#         "False -> otherwise.\n"
#         "No extra text."
#     )

#     # question = "Is the rule which equivalent to the " + new_intent + "; found in the " + topology_file + " configuration file ?, answer True for yes, False for no. " 
#     # print(question)
#     context_variables.update({
#               "topology_file" : config_text,  # Path to the folder containing the configuration file
#                    "question" : conflict_q        # User query
#                         })
#     rule_exists = Answer_Query(context_variables)   
#     rule_exists = str(rule_exists).strip().lower() == "true"
#     print("rule_exists:", rule_exists)

#     rule_line = None
#     if rule_exists:
#         line_q = (
#             "From the Cisco IOS configuration text, extract the EXACT single ACL rule line "
#             f"that is semantically equivalent to this intent:\n{new_intent}\n\n"
#             "Return ONLY that single rule line exactly as written. "
#             "If you cannot find an exact line, return None (only the word None)."
#         )
#         context_variables.update({
#             "topology_file" : config_text,  
#                "question" : line_q        # User query
#                     })
#         rule_line_raw = Answer_Query(context_variables) 
#         rule_line = None if _norm_nl(rule_line_raw) == "none" else rule_line_raw
#         print("Matched rule line:", rule_line)
#     else:
#         print("Rule is not exist (per agent)")
        
#     ########################### Step 4: ACL Placement Resolver Agent: interface, direction, existing ACL name on that interface ##########################
#     ### before adding a rule, we must determine:
#         # 1- Where does the traffic enter the router?
#         # 2- In which direction do we want to filter it?
#         # 3- Is there already an ACL applied there?
#     ### then you can pick the correct ACL name or create a new one
#     #######################################################################################################################
    
#     topo_prompt = network_topology_json if network_topology_json is not None else topo
#     topo_prompt_text = json.dumps(topo_prompt)

#     intf_q = (
#         f"Topology (JSON): {topo_prompt_text}\n\n"
#         f"Intent: {new_intent}\n"
#         f"Chosen device: {hostname}\n\n"
#         "Question: On which interface of the chosen device does the described traffic ENTER the router?\n"
#         "Return interface name only (example: g0/0). No extra text."
#     )
#     context_variables.update({
#         "topology_file": topo_prompt_text,
#         "question": intf_q
#     })
#     Intf_Name = Answer_Query(context_variables)                  

#     print("Router interface where the described intent enter the router is :")
#     print(Intf_Name)
#     # print("Ingress interface:", Intf_Name)


#     # question = (
#     # "Based on the user intent " + new_intent +
#     # " and the topology " + json.dumps(network_topology_json) +
#     # ", on which router interface does the described intent enter the router? (Provide interface name only.)"
#     # )
#     dir_q = (
#         f"Intent: {new_intent}\n"
#         f"Chosen device: {hostname}\n"
#         f"Ingress interface: {Intf_Name}\n\n"
#         "Should the ACL be applied inbound or outbound on that interface?\n"
#         "Return ONLY: in OR out"
#     )
#     context_variables.update({
#         "topology_file": topo_prompt_text,
#         "question": dir_q
#     })
#     direction = Answer_Query(context_variables)                  
    
#     print("Direction where the ACL will be applied on that interface:")

#     direction = "in" if direction.lower().startswith("in") else ("out" if direction.lower().startswith("out") else direction)
#     print("direction:", direction)

#     # question = (
#     # "Based on the user intent " + new_intent +
#     # "should the ACL be applied in the inbound or outbound direction on that interface? (Answer: in / out.)"
#     # )

#     acl_q = (
#         "You are reading a Cisco IOS router configuration.\n\n"
#         f"Interface: {Intf_Name}\n"
#         f"Direction: {direction}\n\n"
#         "If an ACL is applied on that interface in that direction, return the ACL NAME only.\n"
#         "If no ACL is applied, return none.\n"
#         "Return ONLY one token (ACL name or none)."
#     )
#     context_variables.update({
#         "topology_file": config_text,
#         "question": acl_q
#     })

#     ACLname_raw = Answer_Query(context_variables)
    
#     ACLname = to_none_if_noneish(ACLname_raw)

#     if ACLname is None:
#         List_Found = False
#         print("No ACL applied on that interface/direction.")
#     else:
#         List_Found = True
#         print("ACL applied:", ACLname)
        

#     # question = (
#     # f"According to the current router configuration {topology_file}, "
#     # f"is there already an ACL applied on interface {Intf_Name} "
#     # f"in the {direction} direction? "
#     # f"(Answer: if yes, return ACL name only. If no, return none.)"
#     # )
#     # print(question)

#     ########################### Step 5: ACL Generator agent ##########################

#                 ###############################################
#                    # Case A: rule exists AND ACL is applied
#                 ###############################################
#     if rule_exists and ACLname is not None:
#         applied_q = (
#             "You are checking whether an intent is correctly implemented on a specific interface.\n\n"
#             f"Intent: {new_intent}\n"
#             f"Interface: {Intf_Name}\n"
#             f"Direction: {direction}\n"
#             f"ACL name applied: {ACLname}\n\n"
#             "Return ONLY True or False:\n"
#             "True  -> if the ACL applied on that interface/direction contains a rule that enforces the intent.\n"
#             "False -> otherwise.\n"
#             "No extra text."
#         )
# #       question = (
# #         f"Based on the router configuration {topology_file}, "
# #         f"is the intent '{new_intent}' correctly applied on interface {Intf_Name}? "
# #         f"Answer only with True or False.")
#         context_variables.update({
#                 "topology_file" : config_text,  # Path to the folder containing multiple ACL configuration files
#                      "question" : applied_q 
#                         })
#         applied = Answer_Query(context_variables)
#         print("Applied:", applied)
        
#         if applied:
#             # Nothing to do
#             print("The rule already exists and is applied correctly. Exiting.")
#             # configuration_response = None
#             # direction              = None

#             return (
#                 rule_line, hostname, config_text, None, extraction_result,
#                 ACLname, None, Intf_Name, File_name, List_Found
#             )
            
#         # rule exists but NOT applied correctly -> generate config to apply / fix
#         # (No interactive input; auto-generate to apply using existing ACL)
#         print("Rule exists but not applied correctly -> generating config to apply/fix.")
#         context_variables.update({
#                   "mode": "generate",
#              "direction": direction,
#             "List_Found": True,
#                 "L_Name": ACLname,
#              "Intf_Name": Intf_Name,
#                 "src_ip": source_ip,        
#                 "dst_ip": destination_ip,        
#             "src_subnet": src_Subnet,    
#             "dst_subnet": dst_Subnet,    
#               "protocol": protocol,      
#                   "port": port ,        
#                 "action": action , 
#             })
#        ############ Step 5: ACL Agent: Generate the configuration required to add the rule to the configuration file ###########
#         configuration_response = ACL_generator_caller(context_variables)
#         Whole_configuration    = process_configuration(configuration_response)
#         Rules                  = generate_config2_file(file_path, configuration_response, Intf_Name, config_text, File_name)
#         return (
#             Rules, hostname, config_text, configuration_response, extraction_result,
#             ACLname, direction, Intf_Name, File_name, List_Found
#         )
#             #########################################################################
#                # Case B: rule exists BUT no ACL applied on that interface/direction 
#             #########################################################################
#     # not recreate the ACL, not regenerate the rule, not duplicate existing ACL definitions
#     # ONLY apply the existing ACL to the specified interface, using the provided ACL name and the provided interface and direction
#     if rule_exists and ACLname is None:
#         print(rule_exists)
#         print(ACLname is None)
#         print("rule_exists:", rule_exists, type(rule_exists))
#         print("Rule exists in config, but no ACL applied on the target interface/direction -> generate to apply ACL.")
#         context_variables.update({
#                    "mode": "ApplyOnIntf",
#               "direction": direction,
#              "List_Found": True,   
#                  "L_Name": ACLname,
#              "Intf_Name" : Intf_Name,
#         })
#         configuration_response = ACL_generator_caller(context_variables)
#         Whole_configuration = process_configuration(configuration_response)   # for deployment CLI

#         # # attempt to extract ACL name from generated config (optional)
#         # gen_acl_name = extract_ACL_name(Whole_configuration)
#         # ACLname = gen_acl_name or ACLname
#         # List_Found = ACLname is not None
#         apl = generate_config2_file_apply_only(file_path, configuration_response, File_name)  # for config file update (not the CLI wrapper)
#         # Rules = generate_config2_file(configuration_response, Intf_Name, config_text, File_name)
#         Rules = rule_line # RETURN THE EXISTING RULE
        
#         return (
#             Rules, hostname, config_text, configuration_response, extraction_result,
#             ACLname, direction, Intf_Name, File_name, List_Found
#         )
#             #####################################################################################
#                # Case C: rule does NOT exist BUT ACL is applied -> add rule to existing ACL
#             #####################################################################################
        
#     if (not rule_exists) and ACLname is not None:
#         print("ACL is applied but rule does not exist -> add rule to existing ACL.")
#         context_variables.update({
#                   "mode": "generate",
#              "direction": direction,
#             "List_Found": True,
#                 "L_Name": ACLname,
#             "Intf_Name" : Intf_Name,
#                 "src_ip": source_ip,        
#                 "dst_ip": destination_ip,        
#             "src_subnet": src_Subnet,    
#             "dst_subnet": dst_Subnet,      
#               "protocol": protocol,      
#                   "port": port ,        
#                 "action": action ,

#         })
#         configuration_response = ACL_generator_caller(context_variables)
#         Whole_configuration = process_configuration(configuration_response)
#         Rules = generate_config2_file(file_path,configuration_response, Intf_Name, config_text, File_name)
#         return (
#             Rules, hostname, config_text, configuration_response, extraction_result,
#             ACLname, direction, Intf_Name, File_name, List_Found
#         )
#             #####################################################################################
#                # Case D: neither rule exists nor ACL applied -> create new ACL + apply
#             #####################################################################################

#     print("No rule and no ACL applied -> create new ACL + apply.")
#     context_variables.update({
#               "mode": "generate",
#          "direction": direction,
#         "List_Found": False,
#             "L_Name": ACLname,
#         "Intf_Name" : Intf_Name,
#             "src_ip": source_ip,        
#             "dst_ip": destination_ip,        
#         "src_subnet": src_Subnet,    
#         "dst_subnet": dst_Subnet,      
#           "protocol": protocol,      
#               "port": port ,        
#             "action": action ,

#     })
#     # Step 5: Generate the configuration required to add the rule to the configuration file
#     configuration_response = ACL_generator_caller(context_variables)
#     Whole_configuration = process_configuration(configuration_response)
#     print(configuration_response)
#     gen_acl_name = extract_ACL_name(Whole_configuration)  ## maybe used later 
#     ACLname = gen_acl_name or ACLname
#     List_Found = ACLname is not None
#     print("ACLname,List_Found")
#     print(ACLname,List_Found)
#     Rules = generate_config2_file(file_path, configuration_response, Intf_Name, config_text, File_name)
#     print(Rules)
#     return (
#         Rules, hostname, config_text, configuration_response, extraction_result,
#         ACLname, direction, Intf_Name, File_name, List_Found
#     )

In [7]:
# import inspect

# help(extract_device_override)

In [8]:
# # old ################################################## Program Workflow (old before adjust agents + prompts)  #################################################################
# # (1)File Agent --> (2)Query Agent (many times) --> (3)entity extraction Agent --> (4) ACL generator Agent --> (5)generate updated config 
# #
# # File (config2) --> (6)Analyze IPs --> (8) Stage1: BF validation --> (9) Finetuning (if needed) --> (10) Stage2: GNS3 validation 
# #
# # --> (10) Finetuning (if needed) (11) Deployement
# # #######################################################################################################################################

# def run_ACL_workflow(context_variables = context_variables):
#     topology_file      = context_variables.get("topology_file")          # folder path containing .json
#     network_topology   = context_variables.get("network_topology")          # folder path containing .json
#     new_intent         = context_variables.get("new_intent")          # folder path containing .json
#     Eval_GT_File       = context_variables.get("Eval_GT_File")
#     print("User intent is : " + new_intent)
    
#     ############## Step 1: extract entities of the user intent ##############
#     #########################################################################
       
    
#     extraction_result = Entity_Extractor_caller(context_variables)
#     print(extraction_result)
#     source_ip, destination_ip, protocol, port, action, app, src_Subnet, dst_Subnet = extract_entities(extraction_result)
#     print(source_ip, destination_ip, protocol, port, action, app, src_Subnet, dst_Subnet)

#     # Step 1: Find the related router configuration file
#     File_name = File_Finder_caller(context_variables)
#     file_path1 = file_path+'/'+ File_name + '.cfg' #'.json' i changed it 22/1 10:20
#     topology_content = read_topology_file(file_path1)
#     # Optionally, convert the list of lines to a single string (if needed)
#     topology_file = ''.join(topology_content) if topology_content else ""
#     hostname      = File_name #extract_hostname(file_path1)
#     print(File_name)

#     # ## for evaluate Agent 1 ##
#     # Eval_hostname = choose_device_only(source_ip, destination_ip, src_Subnet, dst_Subnet, network_topology)
#     # print("Eval_hostname:")
#     # print(Eval_hostname)
    
#     context_variables.update({
#         "topology_file" : topology_file,  # Path to the folder containing the configuration files
#            # "new_intent" : new_intent # User intent for the new rule
#     })
    
#     question = "is there an existing access list in " + hostname + " configuration file ?" 
#     print(question)
#     context_variables.update({
#           # "topology_file" : topology_file,  # Path to the folder containing the configuration file
#                "question" : question        # User query
#                     })
#     List_Found = Answer_Query(context_variables)                  
#     print(List_Found)
  
#     question = "Is the rule which equivalent to the " + new_intent + "; found in the " + hostname + " configuration file ?" 
#     print(question)
#     context_variables.update({
#           # "topology_file" : topology_file,  # Path to the folder containing the configuration file
#                "question" : question        # User query
#                     })
#     Rule_Found = Answer_Query(context_variables)                   
#     print(Rule_Found)
    
#     if List_Found.lower() == "yes.":
#         question = "what is the name of the existing access list in " + hostname + " configuration file ?" 
#         print(question)
#             # context_variables = {
#             #       "topology_file" : topology_file,  # Path to the folder containing the configuration file
#             #            "question" : question        # User query
#             #         }                   
#             # L_Name = Answer_Query(context_variables)
#             # print(L_Name)
#             # print(topology_file)
#         L_Name = extract_ACL_name(topology_file) # function to extract the ACL name in the config.
#         print("L_Name :" )
#         print(L_Name)
#         if Rule_Found.lower() == "yes" :
           
#             ##############################################################################################################
#             # ################ Case 1: there is ACL and Rule --> check if the rule already applied or not ################
#             ##############################################################################################################
            
#             print("the rule is already existed in the ACL")
#             print("let me check if the rule already applied on the interfaces or not?")

#             question = "On which interface the access list " + L_Name + ", applied in the " + hostname + " configuration file ? If not applied return false." 
#             print(question)
#             context_variables.update({
#                     # "topology_file" : topology_file,  # Path to the folder containing multiple ACL configuration files
#                          "question" : question # User intent for the new rule
#                             })
#             Intf_Name = Answer_Query(context_variables)
#             # print(Intf_Name)
#             if Intf_Name.lower() != "false":
#                 question = "On which direction the access list " + L_Name + ", applied on the " + Intf_Name + " interface in the " + hostname + " configuration file ?" 
#                 print(question)
#                 context_variables.update({
#                         # "topology_file" : topology_file,  # Path to the folder containing the configuration file
#                              "question" : question # User intent for the new rule
#                     })
#                 Dir_Int = Answer_Query(context_variables)
#                 print("Dir_Int:")
#                 print(Dir_Int)
#                 Whole_configuration = None
#                 extraction_result   = None
#                 direction           = None
#                 print("The rule is already existed and applied on the interface. \nThanks for your response! Bye.")
#                 # return Rules , hostname, topology_file, Whole_configuration, extraction_result, L_Name, direction, Intf_Name, File_name, List_Found
#                 return Rules, hostname, topology_file, configuration_response, extraction_result, L_Name, direction, Intf_Name, File_name, List_Found

#             else:     # #### Intf_Name = "false" , we need only to apply the rule on the interface
#                 print("The rule is already existed but not applied on the interface.")
#                 apply_rule = input("Do you want to apply the rule on the interface in the the configuration file " + File_name + " ? (yes/no): ").strip().lower()
#                 if add_new_acl == 'yes':
#                     # Step 4: extract entities of the user intent 
#                     extraction_result = Entity_Extractor_caller(context_variables)
#                     print(extraction_result)
#                     source_ip, destination_ip, protocol, port, action, app, src_Subnet, dst_Subnet = extract_entities(extraction_result)
#                     # user_action = action
#                     # ##### Specify the direction where we will apply the rule
#                     direction = get_direction(topology_file, hostname, source_ip, destination_ip, src_Subnet, dst_Subnet)             
#                     print("You asked to add the rule.")   
                    
#                     context_variables.update({
#                             "mode": "generate",
#                          # "topology_file" : topology_file,  # Path to the folder containing the configuration file
#                             # "new_intent" : new_intent,     # User intent
#                              "direction" : direction,      # the direction of traffic flow for the rule
#                             "List_Found" : List_Found,      # if ACL exist
#                                 "L_Name" : L_Name          # the ACL name
        
#                     })
#                     # Step 6: Generate the configuration required to add the rule to the configuration file
#                     configuration_response = ACL_generator_caller(context_variables)
#                     Whole_configuration = process_configuration(configuration_response)
#                     Rules = generate_config2_file(configuration_response, topology_file, File_name)
#                     question = "On which interface the access list " + L_Name + ", applied in the generated " + configuration_response + " ? If not applied return false." 
#                     print(question)
#                     context_variables.update({
#                             "topology_file" : configuration_response,  # Path to the folder containing multiple ACL configuration files
#                                  "question" : question # User intent for the new rule
#                                     })
#                     Intf_Name = Answer_Query(context_variables)
#                     # return Rules, hostname, topology_file, Whole_configuration, extraction_result, L_Name
#                     return Rules, hostname, topology_file, configuration_response, extraction_result, L_Name, direction, Intf_Name, File_name, List_Found

#                 else: # add_new_acl == 'No'  not to add 
#                     # Whole_configuration = None
#                     configuration_response = None
#                     Intf_Name = None
#                     extraction_result   = None
#                     print("OK, I will not apply the rule. \nThanks for your response! Bye.")
#                     # return hostname, topology_file, Whole_configuration, extraction_result, L_Name
#                     return hostname, topology_file, configuration_response, extraction_result, L_Name, Intf_Name, File_name, List_Found


#             ##############################################################################################################
#             # ################################ Case 2: there is ACL but no Rule  ################################
#             ##############################################################################################################

#         else: #Rule_Found == "false" --> so add the new rule to the list -> not complete
#             # Step 3: extract specific information about the existing access list name, which interface and direction applied on
            
#             print("let's add the rule for the existed defined ACL? maybe the rule deleted by mistakes or not added yet")
#             add_rule = input("Do you want to add the rule to the existed access list in the configuration file " + File_name + "? (yes/no): ").strip().lower()
#             if add_rule == 'yes':
#                 # Step 4: extract entities of the user intent 
#                 extraction_result = Entity_Extractor_caller(context_variables)                    
#                 print(extraction_result)
#                 source_ip, destination_ip, protocol, port, action, app, src_Subnet, dst_Subnet = extract_entities(extraction_result)
#                 # user_action = action
#                 # ##### Specify the direction where we will apply the rule
#                 direction = get_direction(topology_file, hostname, source_ip, destination_ip, src_Subnet, dst_Subnet)
                
#                 # print("You will add the rule.")
#                 # apply_config(hostname, processed_conf)
            
#                 context_variables.update({
#                             "mode": "generate",
#                      "topology_file" : topology_file,  # Path to the folder containing the configuration file
#                         # "new_intent" : new_intent,     # User intent
#                          "direction" : direction,      # the direction of traffic flow for the rule
#                         "List_Found" : List_Found,      # if ACL exist
#                             "L_Name" : L_Name          # the ACL name
    
#                 })
                
#                 # Step 6: Generate the configuration required to add the rule to the configuration file
#                 configuration_response = ACL_generator_caller(context_variables)
#                 Whole_configuration = process_configuration(configuration_response)
#                 Rules = generate_config2_file(configuration_response, topology_file, File_name)
#                 question = "On which interface the access list " + L_Name + ", applied in the generated " + configuration_response + " ? If not applied return false." 
#                 print(question)
#                 context_variables.update({
#                             "topology_file" : configuration_response,  # Path to the folder containing multiple ACL configuration files
#                                  "question" : question # User intent for the new rule
#                                     })
#                 Intf_Name = Answer_Query(context_variables)
#                 # return Rules, hostname, topology_file, Whole_configuration, extraction_result, L_Name
#                 return Rules, hostname, topology_file, configuration_response, extraction_result, L_Name, direction, Intf_Name, File_name, List_Found

#             else: # add_rule == 'No'  not to add 
#                 # Whole_configuration = None
#                 configuration_response = None
#                 direction = None
#                 Intf_Name = None
#                 extraction_result   = None
#                 print("OK, You asked to not add the rule. \nThanks for your response! Bye.")
#                 # return Rules, hostname, topology_file, Whole_configuration, extraction_result, L_Name
#                 return Rules, hostname, topology_file, configuration_response, extraction_result, L_Name, direction, Intf_Name, File_name, List_Found

                
#             ##############################################################################################################
#             # ################################ Case 3: there is no ACL and no Rule  ################################
#             ##############################################################################################################
         
#     else: # no list && no rule
#         # Step 3: extract entities of the user intent 
#         extraction_result = Entity_Extractor_caller(context_variables)
#         source_ip, destination_ip, protocol, port, action, app, src_Subnet, dst_Subnet = extract_entities(extraction_result)
#         # user_action = action
#         # Ask if the user wants to add a new access list
#         ############################################################
#         # Specify the direction where we will apply the rule
#         ############################################################
#         direction = get_direction(topology_file, hostname, source_ip, destination_ip, src_Subnet, dst_Subnet)         
#         print("direction:" + direction)
        
#         add_new_acl = input("Do you want to add a new access list to the configuration file " + File_name + "? (yes/no): ").strip().lower()
#         if add_new_acl == 'yes':
#             print("You have chosen to add a new access list.")
            
#             # apply_config(hostname, processed_conf)
#             context_variables.update({
#                             "mode": "generate",
#                  "topology_file" : topology_file,  # Path to the folder containing multiple ACL configuration files
#                     # "new_intent" : new_intent, # User intent for the new rule
#                      "direction" : direction # the direction of traffic flow for the rule
   
#                 })
#             # print("Direction : "+direction)           
#             # Step 5: Generate the configuration required to add the rule to the configuration file
#             configuration_response = ACL_generator_caller(context_variables)

#             print(configuration_response)
#             ###### find the used ACL name use in the generated config using python function            
#             Whole_configuration = process_configuration(configuration_response)
#             L_Name = extract_ACL_name(Whole_configuration)

#             ### to extract the ACL name from the generated config --> via Query_agent
#             question = "On which interface the access list " + L_Name + ", applied in the generated " + configuration_response + " ? If not applied return false." 
#             print(question)
#             context_variables.update({
#                             "topology_file" : configuration_response,  # Path to the folder containing multiple ACL configuration files
#                                  "question" : question # User intent for the new rule
#                                     })
#             Intf_Name = Answer_Query(context_variables)
#             Rules = generate_config2_file(Whole_configuration, topology_file, File_name)
#             print(Rules)
            
#             # #### generate config 2 (Agent *)**** it works well but I replace it by a python script
#             # context_variables = {
#             #          "topology_file" : topology_file,  # Path to the folder containing multiple ACL configuration files
#             #             "new_intent" : new_intent, # User intent for the new rule
#             #               "hostname" : hostname, # the hostname
#             #               "commands" : Whole_configuration
#             #         }
#             # config_genre = client.run(
#             #                         agent = Config_Agent,
#             #                      messages = [{"role": "system", "content": new_intent}],
#             #             context_variables = context_variables,
#             #     )
#             # config_response = config_genre.messages[-1]["content"]
#             # print("config2 = " + config_response)
            
#             # return Rules, hostname, topology_file, Whole_configuration, extraction_result, L_Name
#             return Rules, hostname, topology_file, configuration_response, extraction_result, L_Name, direction, Intf_Name , File_name, List_Found

#         else:
#             # Whole_configuration = None
#             configuration_response = None
#             direction             = None
#             extraction_result   = None
#             Intf_Name           = None
#             L_Name              = None
#             Rules               = None
#             print("OK, I will not add the rule. \nThanks for your response! Bye.")
#             # return Rules, hostname, topology_file, Whole_configuration, extraction_result, L_Name
#             return Rules, hostname, topology_file, configuration_response, extraction_result, L_Name, direction, Intf_Name, File_name, List_Found


In [10]:
# if you need to test Batfish or GNS3 with no need to generate the rule --> activate this 
Rule = 'permit ip host 172.16.30.3 any' # based on the intent you test
hostname = 'R2' # based on the device where you apply the intent you test
L_Name = 'ACL_F1_0_IN'
# ######## change the entities based on your intent ###########
source_ip = "172.16.30.3"
destination_ip = None
protocol = None 
port = None 
action = "allow"
app = None 
src_Subnet = "172.16.30.0"
dst_Subnet = None

Intf_Name = "FastEthernet1/0"
direction = "in"

# debugging
# configuration_response = "```\nip access-list extended ACL_F1_0_IN\n permit ip host 172.16.30.3 any\ninterface FastEthernet1/0\n ip access-group ACL_F1_0_IN in\n```"
# configuration_response
# Whole_configuration


## test GNS3 only
configuration_response = "```\nip access-list extended ACL_F1_0_IN\n permit ip host 172.16.30.3 any\ninterface FastEthernet1/0\n ip access-group ACL_F1_0_IN in\n```"
# # Whole_configuration
# hostname = 'R2' 

In [5]:
########################### STAGE 1 : Verification using Batfish ###################################
##### Batfish : 3-step process that uses Batfish to make provably safe and correct changes to ACLs and firewall rules
#### A primary use case for Batfish is to validate configuration changes before deployment (though it can be used to validate deployed configurations as well).
#### Batfish does NOT require direct access to network devices. The core analysis requires only the configuration of network devices. 
####################################################################################################

In [9]:

# configuration_response = "```\nip access-list extended ACL_F1_0_OUT \n permit ip host 172.16.30.3 any\n interface f1/0\nip access-group ACL_F1_0_OUT out```"

In [24]:
context_variables = {
    # Intent & rule
    "new_intent" : new_intent,#"Permit only the host 172.16.30.3 from exiting router R2",
    # "rule": Rule[0],  # the generated ACL line or config snippet
    # "hostname"   : "R2",            # device under test (must match Batfish node)
    "L_Name"     : ACLname,#"acl_in",#L_Name,#"acl_in",            # ACL name
    "extraction_result" : extraction_result,
    # Extracted entities (from your extractor)
    "List_Found" : List_Found ,#"yes"
    "Intf_Name"  : Intf_Name,#"FastEthernet1/0",#Intf_Name, #"FastEthernet1/0",#"Serial0/1",
    "direction"  : direction, #"in",#direction, #
    "File_name"  : File_name,#"R2",#hostname,
    # Network address space (VERY IMPORTANT)
    "Sub_Net"    : Sub_Net,
    "Rule"       : Rules,#"permit ip host 172.16.30.3 any",#Rules[0],
    # Optional but recommended
    "file_path"  : file_path,#"configs/R2_2.cfg",
    "snapshot_name" : "configs",
    "configuratio|n_response" : configuration_response,
}

In [20]:
source_ip, destination_ip, protocol, port, action, app, src_Subnet, dst_Subnet = extract_entities(extraction_result)
ace_hint = None
for line in configuration_response.splitlines():
    s = line.strip()
    if s.startswith(("permit ", "deny ")):
        ace_hint = s
        break
        
# Rule = ' deny tcp 172.16.10.0 0.0.0.255 172.16.50.0 0.0.0.255 eq 80' # based on the intent you test
# File_name = 'R3' # based on the device where you apply the intent you test
# L_Name = 'ACL_S0_0_IN'
# # ######## change the entities based on your intent ###########
# source_ip = None
# destination_ip = None
# protocol = "tcp" 
# port = "80" 
# action = "allow"
# src_Subnet = "172.16.10.0"
# dst_Subnet = "172.16.50.0"

# Intf_Name = "s0/0"
# direction = "in"

intent = ACLRuleIntent(
    intent_text=new_intent,
    action=action,
    protocol=protocol,
    src_ip=source_ip,
    dst_ip=destination_ip,
    src_subnet=src_Subnet,
    dst_subnet=dst_Subnet,
    port=port,
    acl_name=ACLname,
    router_name=File_name,
    interface=Intf_Name,
    direction= direction,
    expected_ace_hint=ace_hint,#Rules,
)

# configuration_response = "```\nip access-list extended ACL_S0_1_OUT \n permit icmp host 172.16.10.3 host 172.16.30.3\n interface s0/1\nip access-group ACL_S0_1_OUT out```"
# acl_commands = [
#     "enable",
#     "configure terminal",
#     "ip access-list extended ACL_LAN_IN",
#     "10 deny icmp host 192.168.10.10 host 192.168.20.10",
#     "20 permit ip any any",
#     "interface g0/0",
#     "ip access-group ACL_LAN_IN in",
#     "end",
#     "write memory",
# ]
Whole_configuration = process_configuration(configuration_response)
# acl_commands = [line.strip() for line in Whole_configuration.splitlines() if line.strip()]
acl_commands = [line.strip() for line in configuration_response.splitlines() if line.strip()]
report = validate_acl_in_gns3(
                      intent= intent,
                acl_commands= acl_commands,
                      consol= Consol,
                     sub_net= Sub_Net,
                include_negative_tests=True,
)

print_summary(report)
save_report_json(report, os.path.join(OUTPUT_DIR, "acl_validation_report.json"))
save_report_csv(report, os.path.join(OUTPUT_DIR, "acl_validation_report.csv"))

VERIFY OUT:

$ terminal length 0
terminal length 0
R2#

$ show access-lists
show access-lists
Extended IP access list ACL_F1_0_IN
    10 permit ip host 172.16.30.3 any
    20 deny ip host 172.16.30.3 any
Extended IP access list ACL_S0_1_OUT
    10 permit icmp host 172.16.10.3 host 172.16.30.3
R2#

$ show running-config | include access-group
show running-config | include access-group
 ip access-group ACL_S0_1_IN in
 ip access-group ACL_S0_1_OUT out
 ip access-group ACL_F1_0_IN in
R2#
PRECHECK OUT:

$ terminal length 0
terminal length 0
R2#

$ show ip interface brief
show ip interface brief
Interface                  IP-Address      OK? Method Status                Protocol
FastEthernet0/0            unassigned      YES DHCP   up                    up      
Serial0/0                  172.16.40.1     YES NVRAM  up                    up      
FastEthernet0/1            10.133.14.137   YES DHCP   up                    up      
Serial0/1                  172.16.20.2     YES NVRAM  up       

In [12]:
### Batfish Finetunnig ###
# Q1 
# def Q1_Finetuning (extraction_result, topology_file, hostname, source_ip, destination_ip, src_Subnet, dst_Subnet ):
#     source_ip, destination_ip, protocol, port, action, app, src_Subnet, dst_Subnet = extract_entities(extraction_result)
#     direction = get_direction(topology_file, hostname, source_ip, destination_ip, src_Subnet, dst_Subnet)         
#     context_variables = {
#                      "topology_file" : topology_file,  # Path to the folder containing multiple ACL configuration files
#                         "new_intent" : new_intent, # User intent for the new rule
#                          "direction" : direction # the direction of traffic flow for the rule
#                     }
#     configuration_response = client.run(
#                                     agent = ACL_agent,
#                                  messages = [{"role": "system", "content": new_intent}],
#                         context_variables = context_variables,
#                     )
#     configuration_response = configuration_response.messages[-1]["content"]
#     print(configuration_response)
#                 ###### find the used ACL name use in the generated config using python function            
#     Whole_configuration = process_configuration(configuration_response)
#     L_Name = extract_ACL_name(Whole_configuration)
#                 ### to extract the ACL name from the generated config --> via Query_agent
                
#     Rules = generate_config2_file(Whole_configuration, topology_file, File_name)
#     print(Rules)
#     return topology_file, Whole_configuration, L_Name

In [8]:
# tests = generate_acl_tests(source_ip, destination_ip, Sub_Net)

# results = run_acl_tests(tests, action, Consol)

# for r in results:
#     print(r)

ping 172.16.30.2
Cannot resolve ping

PC4>
ping 172.16.10.2

84 bytes from 172.16.10.2 icmp_seq=1 ttl=62 time=19.533 ms
84 bytes from 172.16.10.2 icmp_seq=2 ttl=62 time=5.898 ms
84 bytes from 172.16.10.2 icmp_seq=3 ttl=62 time=5.940 ms
84 bytes from 172.16.10.2 icmp_seq=4 ttl=62 time=6.171 ms
84 bytes from 172.16.10.2 icmp_seq=5 ttl=62 time=6.494 ms

{'src_pc': 'PC4', 'dst_ip': '172.16.30.2', 'command': 'ping 172.16.30.2', 'expected': 'allow', 'success': False, 'pass': True}
{'src_pc': 'PC4', 'dst_ip': '172.16.10.2', 'command': 'ping 172.16.10.2', 'expected': 'allow', 'success': False, 'pass': True}


In [22]:
print(out2)

NameError: name 'out2' is not defined

In [23]:
# def choose_src_dst(sub_net):
#     # Randomly choose two unique PCs for the first pair
#     first_src, first_dst = random.sample(sub_net, 2)

#     while True:
#         # Randomly choose two PCs for the second pair
#         second_src, second_dst = random.sample(sub_net, 2)
#         # Ensure the src and dst are not the same as the first pair's src and dst
#         if first_src != second_src or first_dst != second_dst:
#             break

#     return (first_src, first_dst), (second_src, second_dst)

# # Choose the PCs
# (first_src, first_dst), (second_src, second_dst) = choose_src_dst(Sub_Net)

# # Print the results
# print(f"First Pair -> Src: {first_src['D_Name']} ({first_src['D_IP']}), Dst: {first_dst['D_Name']} ({first_dst['D_IP']})")
# print(f"Second Pair -> Src: {second_src['D_Name']} ({second_src['D_IP']}), Dst: {second_dst['D_Name']} ({second_dst['D_IP']})")

In [24]:
tst_command[0]#.get("commands")#.get("device")

NameError: name 'tst_command' is not defined

In [25]:
context_variables = {
    "router_config " : topology_file,  # Path to the folder containing multiple configuration files
          "acl_rule" : rule,  # the new rule added
        "new_intent" : new_intent, # the intent
          "Sub_Net"   : Sub_Net, # PCs info
}

troubleshooting_response = client.run(
                        agent = Troubleshoot_Agent,
                     messages = [{"role": "user", "content": file_path}],
            context_variables = context_variables,
    )
troubleshooting_Rspns = troubleshooting_response.messages[-1]["content"]
print(troubleshooting_Rspns)

NameError: name 'rule' is not defined

In [26]:
new_intent

'Permit only the host 172.16.30.3 from exiting router R2'

In [120]:
# def extract_IP_info(input_string):
#     extracted_data = {}

#     # Define the patterns for extracting key-value pairs
#     patterns = {
#         'src_IP': r'- src_IP_Address: (\S+)',
#         'src_type': r'- src_type: ([\w\s]+)',
#         'src_suggested Address': r'- src_suggested Address: (\S+)',
#         'dst_IP': r'- dst_IP_Address: (\S+)',
#         'dst_type': r'- dst_type: ([\w\s]+)',
#         'dst_suggested Address': r'- dst_suggested Address: (\S+)',
#     }

#     # Search for each pattern in the input string and store the result
#     for key, pattern in patterns.items():
#         match = re.search(pattern, input_string)
#         if match:
#             extracted_data[key] = match.group(1)
#         else:
#             # In case of no match, add the default value
#             if "None" in key:
#                 extracted_data[key] = "None"
#             else:
#                 extracted_data[key] = "any"
#     return extracted_data

In [301]:
change_traffic = HeaderConstraints(srcIps= "172.16.30.3", #172.16.30.3
                                   dstIps= dst_suggested)
                                   # ipProtocols= protocol)#, #["tcp"],
                                   # dstPorts= port) #"80, 8080")

answer = bf.q.testFilters(headers = change_traffic,
                           filters = L_Name,#"acl_in",
                           nodes = hostname
                           ).answer(   #action="permit").answer(
                            snapshot=SNAPSHOT_NAME)
show(answer.frame())

,Node,Filter_Name,Flow,Action,Line_Content,Trace


In [ ]:
change_traffic = HeaderConstraints(srcIps= "172.16.30.2", #172.16.30.3
                                   dstIps= dst_suggested)
                                   # ipProtocols= protocol)#, #["tcp"],
                                   # dstPorts= port) #"80, 8080")

answer = bf.q.testFilters(headers = change_traffic,
                           filters = L_name,#"acl_in",
                           nodes = hostname
                           ).answer(   #action="permit").answer(
                            snapshot=SNAPSHOT_NAME)
show(answer.frame())

In [ ]:
# line_content = test_filter_output['Line_Content'].iloc[0]  # Get the first row's content
# What if have multiple lines ?? i need to find the rule
# Action_content = test_filter_output['Action'].iloc[0]
if result_empty:
    print(" the rule has not been added to the list")
    #In case 1st step of validation (TestFilter) returns error (== not find the rule) :
    # Go to step 2 and 3 , so on
    ############ Step 2: Configuration Generator ############
    ########## Step 3: check if the rule already existed ##########
else:
    for index, row in test_filter_output.iterrows():
        line_content = row['Line_Content']
        action_content = row['Action']
    # Compare the rule with the line content
        if rule == line_content and action.lower() == action_content.lower() :
            match = True
            print(f"Match found at index {index}: The rule matches the line content and the action!\n the flow is {action}ed from {src_suggested} to {dst_suggested}")
            test_filter_output['Line_Content'].iloc[index]
        else:
            match = False
            print(f"Match not found at index {index}.")
    # print(match)

In [ ]:
print(src_Subnet)
print(dst_Subnet)

In [ ]:
file_path

In [ ]:
# check_Reachibility(hostname)

In [ ]:
### this cell is temporary for test
# 172.16.30.3 None None None Permit None 172.16.30.0 None
# rule = "permit ip host 172.16.30.3 any"
# L_name = "acl_in"

# source_ip = src_ip = "172.16.30.3"
# destination_ip = dst_ip = None
# protocol = None
# port = None 
# action = "Permit" 
# app = None 
# src_Subnet = "172.16.30.0"
# dst_Subnet = None


In [ ]:
# print(source_ip)
# print(destination_ip)

# ############# Step 5:  Analyzing the IP's in the rule #############
# Analyze_response = client.run( 
#     agent=IPAnalysis_Agent,
#     messages=[{"role": "user", "content": ""}],
#     context_variables={"source_ip": source_ip, "destination_ip": destination_ip}
# )
# Analyze_response = Analyze_response.messages[-1]["content"].strip()
# Analyze_response

In [ ]:
# # Extracted information
# extracted_info = extract_IP_info(Analyze_response)
# # print(extracted_info)
# src_IP = extracted_info.get("src_IP")
# src_type = extracted_info.get("src_type")
# src_suggested = extracted_info.get("src_suggested Address")
# dst_IP = extracted_info.get("dst_IP")
# dst_type = extracted_info.get("dst_type")
# dst_suggested = extracted_info.get("dst_suggested Address")

# print(dst_suggested)

# Print the result in key-value format
# for key, value in extracted_info.items():
#     print(f"{key} = {value}")
# print(dst_suggested)

In [ ]:
# file_path     = "./configs" 
# file_contents = read_all_vpc_files_in_folder(file_path)

# context_variables = {
#     "folder_path" : file_path,  # Path to the folder containing multiple configuration files
#     "topology_file" : file_contents,  # Path to the folder containing multiple ACL configuration files
# }
    
# PC_IP_response = client.run(
#                         agent = PC_IP_Agent,
#                      messages = [{"role": "user", "content": file_path}],
#             context_variables = context_variables,
#     )
# Sub_Net = PC_IP_response.messages[-1]["content"]
# print(Sub_Net)

In [ ]:
# # Extract each JSON object using regex
# matches = re.findall(r'{[^}]+}', Sub_Net)

# # Parse each extracted string into a Python dictionary
# json_objects = [json.loads(match) for match in matches]

# # Output the formatted JSON array
# print(json.dumps(json_objects, indent=4))

In [27]:
############### Fintuning function: ......... ###############
#############################################################
def FineTune_ACL_LLM(context_variables):
    applied_commands = context_variables.get("applied_commands", None)
    error_details    = context_variables.get("error_details", None)
    
    return f"""
        You are a network troubleshooting assistant specializing in ACL configuration for routers. Your task is to analyze the following context:
        
        Applied Commands:
        {applied_commands}
        
        Error Details:
        {error_details}
        
        Based on the provided commands and errors, identify the misconfiguration and provide precise steps to fix it. Ensure the steps include:
        - The exact commands to remove incorrect configurations.
        - The exact commands to apply the correct ACL configuration.
        - Verifying the applied configuration with appropriate `show` commands.
        
        Keep your response concise and focused on correcting the error while aligning with best practices for ACL configurations.
        print the output without details or explaination.
    """

##############
## Agent (9) #
##############
DT_tuning_Agent = Agent(
    name="DT_Tuning Agent",
    instructions=FineTune_ACL_LLM,
)

error_details = """ ping 172.16.50.2

172.16.50.2 icmp_seq=1 timeout
172.16.50.2 icmp_seq=2 timeout
172.16.50.2 icmp_seq=3 timeout
172.16.50.2 icmp_seq=4 timeout
172.16.50.2 icmp_seq=5 timeout
"""
context_variables = {
    "applied_commands" : commands,
    "error_details " : error_details,  
}

tuning_response = client.run(
                        agent = DT_tuning_Agent,
                     messages = [{"role": "user", "content": commands}],
            context_variables = context_variables,
    )
tuning_Rspns = tuning_response.messages[-1]["content"]
print(tuning_Rspns)

NameError: name 'commands' is not defined

In [124]:
# # Check and replace None values with an IP from the topology
# scr_suggested, dst_suggested, typed_src, typed_dst = check_and_replace_ip(src_ip, dst_ip, topology_file)

# # Print the results
# print(f"Updated suggested src_IP: {scr_suggested}")
# print(f"Updated suggested dst_IP: {dst_suggested}")
# # print(typed_src, typed_dst)

NameError: name 'src_ip' is not defined

In [ ]:
##########################################################
#################### Helping function ####################
######### to insert a rule to the list : not used ########
##########################################################

# def insert_acl_rule(acl_name, topology_file, new_rule):
#     acl_pattern = re.compile(r"(\d+)\s+(deny|permit)\s+.*")  # Pattern to match ACL rules
#     acl_list_start_pattern = re.compile(r"ip\s+access-list(?:\s+\S+)?\s+(\S+)", re.IGNORECASE)  # Pattern to match any ACL name
#     acl_list_end = False  # Flag to track the end of the ACL block

#     with open(topology_file, 'r') as file:
#         lines = file.readlines()

    # # Find the specified ACL block
    # acl_start_index = None
    # for index, line in enumerate(lines):
    #     match = acl_list_start_pattern.match(line.strip())
    #     if match:
    #         current_acl_name = match.group(1)  # Extract the ACL name
    #         if current_acl_name == acl_name:  # Check if this is the ACL we want
    #             acl_start_index = index + 1  # Start after the "ip access-list ..." line
    #             break

    # if acl_start_index is None:
    #     print(f"ACL block '{acl_name}' not found.")
    #     return

    # print(f"Detected ACL name: {acl_name}")

    # # Parse the ACL entries and find their line numbers
    # acl_entries = []
    # for index in range(acl_start_index, len(lines)):
    #     if not lines[index].strip() or lines[index].strip().startswith("interface"):
    #         acl_list_end = index
    #         break

    #     match = acl_pattern.match(lines[index].strip())
    #     if match:
    #         rule_number = int(match.group(1))  # Get the rule number
    #         acl_entries.append((rule_number, index))

    # # Extract the rule number from the new rule
    # new_rule_number = int(re.match(r"(\d+)", new_rule).group(1))

    # # Find the correct insertion point
    # insert_index = acl_list_end  # Default to end of the ACL block
    # for rule_number, line_index in acl_entries:
    #     if new_rule_number < rule_number:
    #         insert_index = line_index
    #         break

    # # Insert the new rule at the correct position
    # lines.insert(insert_index, "  " + new_rule + "\n")  # Add proper indentation

    # # Write the updated configuration back to the file
    # with open(topology_file, 'w') as file:
    #     file.writelines(lines)

    # print(f"New rule inserted at line {insert_index + 1} inside the ACL block '{acl_name}'.")

In [ ]:
##########################################################
#################### Helping function ####################
######### to delete a rule from the list : not used ########
##########################################################

# def delete_acl_rule(topology_file, rule_to_delete):
#     acl_pattern = re.compile(r"(\d+)\s+(deny|permit)\s+.*")  # Pattern to match ACL rules
#     acl_list_start_pattern = re.compile(r"ip\s+access-list(?:\s+\S+)?\s+(\S+)", re.IGNORECASE)  # Pattern to match ACL name
#     acl_list_end = False  # Flag to track the end of the ACL block

#     with open(topology_file, 'r') as file:
#         lines = file.readlines()

#     # Find the start of any ACL list
#     acl_start_index = None
#     acl_name = None
#     for index, line in enumerate(lines):
#         match = acl_list_start_pattern.match(line.strip())
#         if match:
#             acl_name = match.group(1)  # Extract the ACL name
#             acl_start_index = index + 1  # Start after the "ip access-list ..." line
#             break

#     if acl_start_index is None:
#         print("ACL block not found.")
#         return

#     print(f"Detected ACL name: {acl_name}")

#     # Find the rule to delete and its line number
#     rule_found = False
    # for index in range(acl_start_index, len(lines)):
    #     if not lines[index].strip() or lines[index].strip().startswith("interface"):
    #         acl_list_end = index
    #         break

    #     # Check if the current line matches the rule to delete
    #     if lines[index].strip() == rule_to_delete:
    #         rule_found = True
    #         del lines[index]  # Delete the rule line
    #         print(f"Deleted rule: {rule_to_delete} at line {index + 1}.")
    #         break

    # if not rule_found:
    #     print(f"Rule '{rule_to_delete}' not found in the ACL.")
    #     return

    # # Write the updated configuration back to the file
    # with open(topology_file, 'w') as file:
    #     file.writelines(lines)

    # print("ACL updated successfully.")

In [ ]:
### config is not add it as it is to the config file 
# after check that everything is OK , show the config file of the file 

In [ ]:
# sometimes , it generates the ACL using out instead of in ####

In [ ]:
# Source IP Subnet sometimes without CIDR ???
# The general rule:
#  Standard ACLs do not specify destination addresses, so they should
# be placed as close to the destination as possible.
#  Put the extended ACLs as close as possible to the source of the traffic
# denied. 

In [ ]:
print(source_ip, destination_ip, protocol, port, action, app, src_Subnet, dst_Subnet)

In [ ]:
### this step in case we created a new list or added a rule to the list
# suppose i send the conf to GNS3 and apply them , and return the new conf file 
# updating the conf file
# we used "show running-config", so we test it using Batfish
## Now I have the files
# going to Batfish Questions

In [ ]:
file_path = "./configs"  # Replace with your folder path
file_path = file_path+'/'+File_name
topology_content = read_topology_file(file_path)
# print(topology_content)

# Optionally, convert the list of lines to a single string (if needed)
topology_file = ''.join(topology_content) if topology_content else ""

In [ ]:
search_flow = HeaderConstraints(
        srcIps= scr_suggested,
        dstIps= dst_suggested,
        ipProtocols= protocol
        # dstPorts= port
    )
answer_search = bf.q.searchFilters(
        headers= search_flow,
        action= action,
        nodes=hostname, #hostname
        filters= L_name #L_name
    ).answer(snapshot=SNAPSHOT_NAME)
# Analyze the results from searchFilters
    # empty--> policy is correctly implemented
show(answer_search.frame())

In [ ]:
result = bf.q.ipOwners().answer().frame()

In [ ]:
result

In [ ]:
result = bf.q.filterLineReachability().answer().frame()
result